# EX03. [추가연습] 불균형 데이터 실무전용 문제 모음 - 고객 이탈

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: EX03_불균형데이터_실전문제_고객이탈.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

EX03 이론 노트북과 같은 데이터를 쓰지만, **다른 모델과 임계값 튜닝**으로 새로운 문제를 풀어봅니다.

## 목차
1. 모델 학습 연습
2. 임계값 튜닝 연습
3. PR Curve 연습
4. 딥러닝 + SMOTE 연습

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype(float)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df = df.drop(columns=['customerID'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_train.shape

## 1. 모델 학습 연습

**문제 1.** `XGBClassifier(n_estimators=100, random_state=42, scale_pos_weight=3)`를 학습시켜 recall을 구하세요. (scale_pos_weight는 XGBoost의 불균형 대응 옵션입니다)

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, precision_score
xgb = XGBClassifier(n_estimators=100, random_state=42, scale_pos_weight=3, eval_metric='logloss')  # scale_pos_weight: 양성(소수) 클래스의 가중치를 높여 class_weight='balanced'와 비슷한 효과를 줌(XGBoost 전용 옵션)
xgb.fit(X_train_s, y_train)
pred = xgb.predict(X_test_s)
print(recall_score(y_test, pred), precision_score(y_test, pred))
```

</details>

## 2. 임계값 튜닝 연습

**문제 2.** `LogisticRegression`의 예측 확률에서 임계값을 0.3으로 낮췄을 때와 0.5일 때 recall을 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.linear_model import LogisticRegression
logi = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
proba = logi.predict_proba(X_test_s)[:, 1]  # 임계값을 조정하려면 0/1 예측이 아니라 확률이 필요
pred_05 = (proba > 0.5).astype(int)
pred_03 = (proba > 0.3).astype(int)  # 임계값을 낮추면 더 쉽게 양성(이탈)으로 판단해 recall이 오르는 경향이 있음
print(recall_score(y_test, pred_05), recall_score(y_test, pred_03))
```

</details>

## 3. PR Curve 연습

**문제 3.** PR curve를 그리고 average_precision_score를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import precision_recall_curve, average_precision_score
precision, recall, _ = precision_recall_curve(y_test, proba)  # 임계값을 바꿔가며 계산한 precision/recall 쌍들을 반환
plt.plot(recall, precision)
plt.title(f'AP={average_precision_score(y_test, proba):.3f}')  # AP(Average Precision)는 PR curve 아래 면적을 하나의 숫자로 요약
plt.show()
```

</details>

## 4. 딥러닝 + SMOTE 연습

**문제 4.** SMOTE로 오버샘플링한 뒤 이진분류 DL 모델을 학습시켜 recall을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
import tensorflow as tf
tf.random.set_seed(100)  # 가중치 초기화가 무작위라 결과가 매번 조금씩 달라질 수 있어 시드를 고정
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
smote = SMOTE(random_state=42)
X_over, y_over = smote.fit_resample(X_train_s, y_train)  # 딥러닝에서도 머신러닝과 동일하게 학습 데이터에만 오버샘플링을 적용
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dropout(0.3))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # 이진분류 출력층: 노드 1개 + sigmoid
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_over, y_over, epochs=50, batch_size=32, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
dl_pred = (model.predict(X_test_s).flatten() > 0.5).astype(int)
print(recall_score(y_test, dl_pred))  # 오버샘플링 전후로 recall이 어떻게 개선됐는지 확인
```

</details>